<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-06-rag-systems/lesson-6.7-graphrag/notebooks/exercises-6_7.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 6.7: GraphRAG & Knowledge Graphs

*Module 6 · Retrieval-Augmented Generation (RAG)*

> Vector RAG finds similar chunks. But what if the answer requires connecting facts across 5 different documents? “Which instructor teaches the module that covers the topic related to the student’s complaint?” — that’s a multi-hop question. GraphRAG extracts entities and relationships into a knowledge graph, then traverses the graph to answer questions that vector search alone cannot.

`Knowledge Graphs` · `Neo4j` · `Entity Extraction` · `Multi-Hop` · `60 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-6.7.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Knowledge Graph From Scratch — Entities + Relations — `01_knowledge_graph.py`
2. Step 2 — Neo4j Integration — Production Graph Database — `02_neo4j_graphrag.py`
3. Step 3 — Hybrid GraphRAG — Vectors + Graph Together — `03_hybrid_graphrag.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai numpy


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Knowledge Graph From Scratch — Entities + Relations

Extract entities and relationships from text using an LLM. Build a graph in Python.

**`01_knowledge_graph.py`** · _Block 1 of 3_

KNOWLEDGE GRAPH FROM SCRATCH — Extract entities + relations with LLM, build graph in Python.


In [ ]:
# KNOWLEDGE GRAPH FROM SCRATCH
# Extract entities + relations with LLM, build graph in Python.
import google.generativeai as genai
import json, os
from collections import defaultdict

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-2.0-flash")

class SimpleKnowledgeGraph:
    def __init__(self):
        self.triples = []  # (subject, relation, object)
        self.adjacency = defaultdict(list)

    def extract_from_text(self, text):
        """Use LLM to extract entity-relation triples."""
        prompt = f"""Extract ALL entity-relationship triples from this text.
Return JSON array of objects with: subject, relation, object.
Keep entities short (1-3 words). Standardize names.

Text: {text}

Return ONLY valid JSON array:"""
        resp = model.generate_content(prompt)
        raw = resp.text.strip()
        if raw.startswith("```"): raw = raw.split("\n",1)[1].rsplit("```",1)[0]
        try:
            triples = json.loads(raw)
            for t in triples:
                triple = (t["subject"], t["relation"], t["object"])
                self.triples.append(triple)
                self.adjacency[t["subject"].lower()].append(triple)
                self.adjacency[t["object"].lower()].append(triple)
            return len(triples)
        except: return 0

    def get_neighbors(self, entity, hops=1):
        """Get all triples within N hops of an entity."""
        visited, result = set(), []
        queue = [(entity.lower(), 0)]
        while queue:
            node, depth = queue.pop(0)
            if node in visited or depth > hops: continue
            visited.add(node)
            for triple in self.adjacency.get(node, []):
                result.append(triple)
                for e in [triple[0].lower(), triple[2].lower()]:
                    if e not in visited: queue.append((e, depth+1))
        return result

    def query(self, question):
        """GraphRAG: extract entities from question, traverse, answer."""
        # Step 1: extract query entities
        resp = model.generate_content(f"Extract key entities from this question. Return as JSON array of strings.\nQuestion: {question}")
        raw = resp.text.strip()
        if raw.startswith("```"): raw = raw.split("\n",1)[1].rsplit("```",1)[0]
        try: entities = json.loads(raw)
        except: entities = question.lower().split()[:3]

        # Step 2: gather graph context
        context_triples = []
        for e in entities:
            context_triples.extend(self.get_neighbors(e, hops=2))
        context = "\n".join(set(f"{s} --[{r}]--> {o}" for s,r,o in context_triples))

        # Step 3: answer with graph context
        answer = model.generate_content(
            f"Answer using ONLY this knowledge graph.\n\nGraph:\n{context}\n\nQuestion: {question}\nAnswer:"
        )
        return {"answer": answer.text.strip(), "triples_used": len(context_triples), "entities": entities}

# ── Build graph ──
kg = SimpleKnowledgeGraph()
print("Building Knowledge Graph...\n")

texts = [
    "Netsetos is an edtech company in Hyderabad offering GenAI, Data Science, and Cloud courses.",
    "The GenAI course costs 14999 rupees, has 14 modules, 146 hours, and requires Python and high school math.",
    "Refund policy allows full refund within 7 days, 50% from 8-30 days, and none after 30 days.",
    "Module 6 covers RAG including vector databases, LangChain, LlamaIndex, and GraphRAG.",
    "Live classes are held daily at 7 PM IST. 5000 students trained with 85% completion rate.",
]
for t in texts:
    n = kg.extract_from_text(t)
    print(f"  Extracted {n} triples from: {t[:50]}...")

print(f"\n  Total triples: {len(kg.triples)}")
print(f"  Sample: {kg.triples[:3]}\n")

# ── Query ──
for q in ["What does Module 6 cover?", "What are the prerequisites for the GenAI course?"]:
    r = kg.query(q)
    print(f"  Q: {q}")
    print(f"  A: {r['answer'][:120]}")
    print(f"  Triples used: {r['triples_used']}, Entities: {r['entities']}\n")


> **What just happened?** The LLM extracted entity-relation triples from 5 documents: (Netsetos, OFFERS, GenAI Course), (GenAI Course, REQUIRES, Python), etc. Queries extracted entities from the question, traversed 2 hops in the graph to gather context, then answered from graph triples. Multi-hop: “What does Module 6 cover?” → find Module 6 → traverse to its topics (RAG, LangChain, GraphRAG) → answer. Vector search can’t do this traversal.


### Step 2 · Neo4j Integration — Production Graph Database

For real-scale GraphRAG, store triples in Neo4j and query with Cypher.

**`02_neo4j_graphrag.py`** · _Block 2 of 3_

NEO4J GRAPHRAG — Production knowledge graph — pip install neo4j google-generativeai


In [ ]:
# NEO4J GRAPHRAG — Production knowledge graph
# pip install neo4j google-generativeai
# Docker: docker run -p 7474:7474 -p 7687:7687 neo4j
from neo4j import GraphDatabase
import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-2.0-flash")

class Neo4jGraphRAG:
    def __init__(self, uri="bolt://localhost:7687", user="neo4j", password="password"):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def add_triple(self, subject, relation, obj):
        with self.driver.session() as s:
            s.run(
                "MERGE (a:Entity {name: $subj}) "
                "MERGE (b:Entity {name: $obj}) "
                "MERGE (a)-[r:RELATION {type: $rel}]->(b)",
                subj=subject, obj=obj, rel=relation
            )

    def query_graph(self, entity, hops=2):
        """Retrieve subgraph within N hops of entity."""
        with self.driver.session() as s:
            result = s.run(
                "MATCH path = (a:Entity {name: $name})-[*1.." + str(hops) + "]->(b) "
                "RETURN a.name AS from, type(relationships(path)[-1]) AS rel, b.name AS to",
                name=entity
            )
            return [(r["from"], r["rel"], r["to"]) for r in result]

    def text_to_cypher(self, question):
        """LLM generates Cypher query from natural language."""
        prompt = f"""Convert this question to a Neo4j Cypher query.
Schema: (:Entity)-[:RELATION {{type: string}}]->(:Entity)
Return ONLY the Cypher query, no explanation.

Question: {question}
Cypher:"""
        resp = model.generate_content(prompt)
        cypher = resp.text.strip().strip('`')
        if cypher.startswith("cypher"): cypher = cypher[6:].strip()
        return cypher

# ── Demo (without Neo4j running, show the pattern) ──
print("Neo4j GraphRAG Pattern:\n")
print("  1. Extract triples from documents (LLM)")
print("  2. Store in Neo4j: MERGE (a)-[:REL]->(b)")
print("  3. Query: natural language -> Cypher -> execute -> results")
print("  4. Feed results to LLM for natural language answer\n")

# Example Cypher queries
examples = [
    ("What does Netsetos offer?",
     "MATCH (n:Entity {name:'Netsetos'})-[:RELATION]->(c) RETURN c.name"),
    ("What are the prerequisites?",
     "MATCH (c:Entity {name:'GenAI Course'})-[:RELATION {type:'REQUIRES'}]->(p) RETURN p.name"),
    ("Multi-hop: What topics are in the course that Netsetos offers?",
     "MATCH (n:Entity {name:'Netsetos'})-[:RELATION]->(c)-[:RELATION {type:'HAS_MODULE'}]->(m) RETURN m.name"),
]
for q, cypher in examples:
    print(f"  Q: {q}")
    print(f"  Cypher: {cypher}\n")


### Step 3 · Hybrid GraphRAG — Vectors + Graph Together

The best approach: vector search for initial retrieval + graph traversal for multi-hop reasoning.

**`03_hybrid_graphrag.py`** · _Block 3 of 3_

HYBRID GRAPHRAG — Vectors + Knowledge Graph


In [ ]:
# HYBRID GRAPHRAG — Vectors + Knowledge Graph
import google.generativeai as genai
import numpy as np, os, json
from collections import defaultdict

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class HybridGraphRAG:
    """Vector retrieval + knowledge graph traversal."""
    def __init__(self):
        self.chunks, self.embs = [], []
        self.kg = SimpleKnowledgeGraph()  # from script 01
        self.model = genai.GenerativeModel("gemini-2.0-flash")

    def ingest(self, text, source="doc"):
        # Vector path: chunk and embed
        emb = np.array(genai.embed_content(model="models/text-embedding-004", content=text)["embedding"])
        self.chunks.append({"text": text, "source": source})
        self.embs.append(emb)
        # Graph path: extract triples
        self.kg.extract_from_text(text)

    def query(self, question):
        # 1. Vector retrieval (semantic similarity)
        qe = np.array(genai.embed_content(model="models/text-embedding-004", content=question)["embedding"])
        scores = [(i, np.dot(qe,e)/(np.linalg.norm(qe)*np.linalg.norm(e))) for i,e in enumerate(self.embs)]
        scores.sort(key=lambda x:-x[1])
        vector_context = "\n".join(self.chunks[i]["text"] for i,_ in scores[:3])

        # 2. Graph traversal (relationship context)
        graph_result = self.kg.query(question)
        graph_context = graph_result["answer"]

        # 3. Combine both contexts
        combined = self.model.generate_content(
            f"Answer using BOTH sources.\n\n"
            f"Vector context:\n{vector_context}\n\n"
            f"Graph context:\n{graph_context}\n\n"
            f"Question: {question}\nAnswer:"
        )
        return {"answer": combined.text.strip(), "method": "hybrid"}

print("Hybrid GraphRAG:")
print("  Vector retrieval finds SIMILAR chunks (content match)")
print("  Graph traversal finds CONNECTED entities (relationship chains)")
print("  Combined: best of both for multi-hop questions")


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
